# Week 4: Window functions

In [1]:
import duckdb

con = duckdb.connect("../sql/taxi_project")

In [5]:
con.sql("""
    SELECT * FROM zone_lookup WHERE LocationID = 1;
""")

┌────────────┬─────────┬────────────────┬──────────────┐
│ LocationID │ Borough │      Zone      │ service_zone │
│   int64    │ varchar │    varchar     │   varchar    │
├────────────┼─────────┼────────────────┼──────────────┤
│          1 │ EWR     │ Newark Airport │ EWR          │
└────────────┴─────────┴────────────────┴──────────────┘

In [9]:
con.sql("""
    SELECT * FROM taxi_clean WHERE pickup_datetime = '2025-01-01 00:18:38';
""")

┌─────────────────────┬─────────────────────┬─────────────────┬───────────────┬────────────────┬────────────────┬─────────────┬────────────┐
│   pickup_datetime   │  dropoff_datetime   │ passenger_count │ trip_distance │ pu_location_id │ do_location_id │ fare_amount │ tip_amount │
│      timestamp      │      timestamp      │      int64      │    double     │     int32      │     int32      │   double    │   double   │
├─────────────────────┼─────────────────────┼─────────────────┼───────────────┼────────────────┼────────────────┼─────────────┼────────────┤
│ 2025-01-01 00:18:38 │ 2025-01-01 00:26:59 │               1 │           1.6 │            229 │            237 │        10.0 │        3.0 │
│ 2025-01-01 00:18:38 │ 2025-01-01 00:23:29 │               1 │          0.85 │            239 │            142 │         7.2 │       3.05 │
└─────────────────────┴─────────────────────┴─────────────────┴───────────────┴────────────────┴────────────────┴─────────────┴────────────┘

*Rank pickup zones by daily revenue*

In [28]:
con.sql("""
    WITH revenue_by_pu_location AS (
            SELECT  
                    DATE(pickup_datetime) AS Date,
                    pu_location_id AS location_id,
                    ROUND(SUM(fare_amount), 2) AS revenue
            FROM taxi_clean
            GROUP BY Date, location_id
        ),
        ranked_location AS (
            SELECT 
                    Date,
                    revenue,
                    location_id,
                    RANK() OVER (
                            PARTITION BY Date
                            ORDER BY revenue DESC
                            ) AS rank
            FROM revenue_by_pu_location
        ) 
        SELECT 
                r.Date,
                z.Zone,
                r.location_id,
                r.revenue,
                r.rank
        FROM ranked_location AS r
        INNER JOIN zone_lookup AS z
        ON r.location_id = z.LocationID
        ORDER BY r.rank, r.Date LIMIT 20;
""")

┌────────────┬─────────────┬─────────────┬───────────┬───────┐
│    Date    │    Zone     │ location_id │  revenue  │ rank  │
│    date    │   varchar   │    int32    │  double   │ int64 │
├────────────┼─────────────┼─────────────┼───────────┼───────┤
│ 2024-12-31 │ JFK Airport │         132 │      73.7 │     1 │
│ 2025-01-01 │ JFK Airport │         132 │ 307853.61 │     1 │
│ 2025-01-02 │ JFK Airport │         132 │ 336847.75 │     1 │
│ 2025-01-03 │ JFK Airport │         132 │ 290531.02 │     1 │
│ 2025-01-04 │ JFK Airport │         132 │ 302722.87 │     1 │
│ 2025-01-05 │ JFK Airport │         132 │ 345352.74 │     1 │
│ 2025-01-06 │ JFK Airport │         132 │ 324399.98 │     1 │
│ 2025-01-07 │ JFK Airport │         132 │ 279517.25 │     1 │
│ 2025-01-08 │ JFK Airport │         132 │ 251281.12 │     1 │
│ 2025-01-09 │ JFK Airport │         132 │ 285841.63 │     1 │
│ 2025-01-10 │ JFK Airport │         132 │ 299390.17 │     1 │
│ 2025-01-11 │ JFK Airport │         132 │ 268385.79 │ 

*Find top 3 zones per borough*

In [47]:
con.sql("""
    WITH revenue AS (
            SELECT
                ROUND(SUM(fare_amount), 2) AS revenue,
                pu_location_id AS location_id
            FROM taxi_clean
            GROUP BY location_id
        ),
        ranked_zones AS (
            SELECT 
                r.location_id,
                r.revenue,
                z.Zone,
                z.Borough,
                ROW_NUMBER() OVER(PARTITION BY z.Borough
                            ORDER BY r.revenue DESC
                        ) AS rank
            FROM revenue AS r
            INNER JOIN zone_lookup as z
            ON r.location_id = z.LocationID
        )
        SELECT 
                rank,
                Zone,
                Borough,
                revenue
        FROM ranked_zones
        WHERE rank <=  3
        ORDER BY Borough, rank;
""")

┌───────┬──────────────────────────────────┬───────────────┬────────────┐
│ rank  │               Zone               │    Borough    │  revenue   │
│ int64 │             varchar              │    varchar    │   double   │
├───────┼──────────────────────────────────┼───────────────┼────────────┤
│     1 │ Co-Op City                       │ Bronx         │    14821.6 │
│     2 │ East Concourse/Concourse Village │ Bronx         │    11881.8 │
│     3 │ Williamsbridge/Olinville         │ Bronx         │    11744.4 │
│     1 │ East New York                    │ Brooklyn      │   41405.08 │
│     2 │ Crown Heights North              │ Brooklyn      │   32672.96 │
│     3 │ Downtown Brooklyn/MetroTech      │ Brooklyn      │   29001.21 │
│     1 │ Newark Airport                   │ EWR           │    5551.42 │
│     1 │ Midtown Center                   │ Manhattan     │ 2203934.01 │
│     2 │ Times Sq/Theatre District        │ Manhattan     │ 1874488.38 │
│     3 │ Upper East Side South       

*Calculate 7-day moving average of trips*

In [61]:
con.sql("""
    WITH daily_trips AS (
            SELECT
                COUNT(*) AS trips,
                DATE(pickup_datetime) AS trip_date
            FROM taxi_clean
            GROUP BY trip_date
        ),
        date_range AS (
            SELECT
                MIN(trip_date) AS first_date,
                MAX(trip_date) AS last_date
            FROM daily_trips
        ),
        calendar AS (
            SELECT
                CAST(day AS DATE) AS pickup_date
            FROM date_range,
            generate_series(first_date, last_date, INTERVAL 1 DAY) AS t(day)
        )
        SELECT
            c.pickup_date,
            COALESCE(d.trips, 0) AS trips,
            ROUND(
                AVG(COALESCE(d.trips, 0)) OVER(
                    ORDER BY c.pickup_date
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ),
                2
            ) AS seven_day_moving_avg
        FROM daily_trips AS d
        LEFT JOIN calendar AS c
        ON c.pickup_date = d.trip_date 
        ORDER BY c.pickup_date
""")

┌─────────────┬────────┬──────────────────────┐
│ pickup_date │ trips  │ seven_day_moving_avg │
│    date     │ int64  │        double        │
├─────────────┼────────┼──────────────────────┤
│ 2024-12-31  │     21 │                 21.0 │
│ 2025-01-01  │  70384 │              35202.5 │
│ 2025-01-02  │  77553 │             49319.33 │
│ 2025-01-03  │  83768 │              57931.5 │
│ 2025-01-04  │  89293 │              64203.8 │
│ 2025-01-05  │  72223 │             65540.33 │
│ 2025-01-06  │  72879 │             66588.71 │
│ 2025-01-07  │  88620 │             79245.71 │
│ 2025-01-08  │  95904 │             82891.43 │
│ 2025-01-09  │  99988 │             86096.43 │
│     ·       │    ·   │                 ·    │
│     ·       │    ·   │                 ·    │
│     ·       │    ·   │                 ·    │
│ 2025-01-23  │ 106262 │             89759.57 │
│ 2025-01-24  │ 101708 │             90608.86 │
│ 2025-01-25  │ 105571 │             92492.86 │
│ 2025-01-26  │  79423 │             927

*Calculate day-over-day revenue change*

In [78]:
con.sql("""
    WITH daily_revenue AS (
            SELECT
                DATE(pickup_datetime) AS date,
                SUM(fare_amount) AS daily_revenue
            FROM taxi_clean
            WHERE fare_amount >= 0
            GROUP BY date
        ),
        date_range AS (
            SELECT
                MIN(date) AS start_date,
                MAX(date) AS end_date
            FROM daily_revenue
        ),
        calendar AS (
            SELECT
                CAST(day AS DATE) AS date
            FROM date_range,
            generate_series(start_date, end_date, INTERVAL 1 DAY) AS t(day)
        ),
        revenue_per_day_lag AS (
            SELECT
                c.date,
                COALESCE(d.daily_revenue, 0) AS revenue,
                LAG(COALESCE(d.daily_revenue, 0)) OVER(
                    ORDER BY c.date
                ) AS prev_revenue
            FROM calendar c
            LEFT JOIN daily_revenue d
            ON c.date = d.date
        )
        SELECT
            date,
            ROUND(revenue, 2) AS revenue,
            ROUND(revenue - prev_revenue, 2) AS daily_revenue_change
        FROM revenue_per_day_lag
        ORDER BY date;
""")

┌────────────┬────────────┬──────────────────────┐
│    date    │  revenue   │ daily_revenue_change │
│    date    │   double   │        double        │
├────────────┼────────────┼──────────────────────┤
│ 2024-12-31 │      401.6 │                 NULL │
│ 2025-01-01 │ 1461385.19 │           1460983.59 │
│ 2025-01-02 │ 1583396.19 │             122011.0 │
│ 2025-01-03 │ 1599759.38 │             16363.19 │
│ 2025-01-04 │ 1655774.41 │             56015.03 │
│ 2025-01-05 │ 1451625.47 │           -204148.94 │
│ 2025-01-06 │ 1395774.39 │            -55851.08 │
│ 2025-01-07 │ 1559595.17 │            163820.78 │
│ 2025-01-08 │ 1639968.59 │             80373.42 │
│ 2025-01-09 │ 1761703.63 │            121735.04 │
│     ·      │      ·     │                ·     │
│     ·      │      ·     │                ·     │
│     ·      │      ·     │                ·     │
│ 2025-01-23 │ 1856008.03 │            162444.25 │
│ 2025-01-24 │ 1775485.46 │            -80522.57 │
│ 2025-01-25 │ 1726620.91 │    

*Calculate percent of total revenue by zone*

In [102]:
con.sql("""
    WITH total_revenue AS (
            SELECT 
                SUM(fare_amount) AS total_revenue
            FROM taxi_clean
            WHERE fare_amount >= 0
        ),
        revenue_by_zone AS (
            SELECT 
                SUM(t.fare_amount) AS zone_total_revenue,
                z.Zone AS zone,
                z.LocationID,
                z.Borough
            FROM taxi_clean AS t
            INNER JOIN zone_lookup AS z
            ON t.pu_location_id = z.LocationID
            GROUP BY zone, LocationID, Borough
        )
    SELECT
        r.zone_total_revenue,
        r.zone,
        ROUND(r.zone_total_revenue / t.total_revenue * 100, 2) AS zone_revenue_percent
    FROM revenue_by_zone AS r
    CROSS JOIN total_revenue AS t
    ORDER BY zone_total_revenue DESC;
""")

┌────────────────────┬─────────────────────────────────────┬──────────────────────┐
│ zone_total_revenue │                zone                 │ zone_revenue_percent │
│       double       │               varchar               │        double        │
├────────────────────┼─────────────────────────────────────┼──────────────────────┤
│  8388507.840000004 │ JFK Airport                         │                16.32 │
│  4467482.080000003 │ LaGuardia Airport                   │                 8.69 │
│ 2203934.0100000026 │ Midtown Center                      │                 4.29 │
│ 1874488.3800000001 │ Times Sq/Theatre District           │                 3.65 │
│ 1775144.1400000018 │ Upper East Side South               │                 3.45 │
│  1723902.650000002 │ Upper East Side North               │                 3.35 │
│ 1663284.7600000007 │ Penn Station/Madison Sq West        │                 3.24 │
│  1538330.140000001 │ Midtown East                        │                

*Find each zone's best revenue day*

In [136]:
con.sql("""
        WITH revenue_per_date AS (
            SELECT
                SUM(t.fare_amount) AS revenue,
                DATE(t.pickup_datetime) AS date,
                z.Zone AS zone,
                z.LocationID AS location,
                z.Borough AS borough
            FROM taxi_clean AS t
            INNER JOIN zone_lookup AS z
            ON t.pu_location_id = z.LocationID
            WHERE t.fare_amount >= 0
            GROUP BY date, zone, location, borough
        ), ranked_revenue AS (
            SELECT
                date,
                zone,
                location,
                borough,
                revenue,
                ROW_NUMBER() OVER(
                    PARTITION BY zone, location, borough
                    ORDER BY revenue DESC, date ASC
                ) AS rank
            FROM revenue_per_date)
        SELECT
            date,
            zone,
            location,
            borough,
            ROUND(revenue, 2) AS best_day_revenue
        FROM ranked_revenue
        WHERE rank = 1
        ORDER BY revenue DESC;
""")

┌────────────┬──────────────────────────────┬──────────┬───────────────┬──────────────────┐
│    date    │             zone             │ location │    borough    │ best_day_revenue │
│    date    │           varchar            │  int64   │    varchar    │      double      │
├────────────┼──────────────────────────────┼──────────┼───────────────┼──────────────────┤
│ 2025-01-20 │ LaGuardia Airport            │      138 │ Queens        │       1027908.06 │
│ 2025-01-20 │ JFK Airport                  │      132 │ Queens        │        391231.61 │
│ 2025-01-30 │ Midtown Center               │      161 │ Manhattan     │        107759.79 │
│ 2025-01-23 │ Upper East Side South        │      237 │ Manhattan     │         84159.74 │
│ 2025-01-15 │ Times Sq/Theatre District    │      230 │ Manhattan     │          78024.3 │
│ 2025-01-09 │ Upper East Side North        │      236 │ Manhattan     │          75106.9 │
│ 2025-01-30 │ Midtown East                 │      162 │ Manhattan     │        

*Compare each trip fare to zone average*

In [169]:
con.sql("""
    WITH zone_avg_fare AS (
        SELECT
            AVG(t.fare_amount) AS zone_avg_fare,
            z.Zone AS zone,
            z.LocationID AS location,
            z.Borough AS borough
        FROM taxi_clean AS t
        INNER JOIN zone_lookup AS z
        ON t.pu_location_id = z.LocationID
        WHERE t.fare_amount >= 0
        GROUP BY 
            zone, 
            location, 
            borough
        )
        SELECT 
            DATE(t.pickup_datetime) AS date,
            z.zone,
            z.location,
            z.borough,
            ROUND(z.zone_avg_fare, 2),
            t.fare_amount AS fare,
            ROUND((t.fare_amount / NULLIF(z.zone_avg_fare, 0) * 100) - 100, 2) AS trip_to_zoneAvg_perc
        FROM zone_avg_fare AS z
        INNER JOIN taxi_clean as t
        ON z.location = t.pu_location_id
        ORDER BY date
""")

┌────────────┬───────────────────────────────┬──────────┬───────────┬───────────────────────────┬────────┬──────────────────────┐
│    date    │             zone              │ location │  borough  │ round(z.zone_avg_fare, 2) │  fare  │ trip_to_zoneAvg_perc │
│    date    │            varchar            │  int64   │  varchar  │          double           │ double │        double        │
├────────────┼───────────────────────────────┼──────────┼───────────┼───────────────────────────┼────────┼──────────────────────┤
│ 2024-12-31 │ West Chelsea/Hudson Yards     │      246 │ Manhattan │                      15.1 │   16.3 │                 7.97 │
│ 2024-12-31 │ Greenwich Village South       │      114 │ Manhattan │                     14.02 │   34.5 │               146.12 │
│ 2024-12-31 │ Central Park                  │       43 │ Manhattan │                     14.45 │   10.7 │               -25.95 │
│ 2024-12-31 │ Penn Station/Madison Sq West  │      186 │ Manhattan │                     

*Find unusually high fares within each zone*

In [220]:
con.sql("""
    WITH fares_per_zones AS (
            SELECT
                DATE(t.pickup_datetime) AS date,
                t.fare_amount AS fare,
                z.Zone AS zone,
                z.LocationID AS location,
                z.Borough AS borough
            FROM taxi_clean AS t
            INNER JOIN zone_lookup AS z
                ON t.pu_Location_id = z.LocationID 
            WHERE t.fare_amount >= 0
        ), 
        zone_stats AS (
            SELECT
                date,
                fare,
                zone,
                location,
                borough,
                AVG(fare) OVER(
                    PARTITION BY zone, location, borough
                ) AS zone_avg_fare,
                STDDEV(fare) OVER(
                    PARTITION BY zone, location, borough
                ) AS zone_std_fare
            FROM fares_per_zones
        )
        SELECT
            date,
            fare,
            zone,
            location,
            borough,
            ROUND(zone_avg_fare, 2) AS zone_avg_fare,
            ROUND(zone_std_fare, 2) AS zone_std_fare,
            ROUND((fare - zone_avg_fare) / NULLIF(zone_std_fare, 0), 2) AS z_score
        FROM zone_stats
        WHERE fare > zone_avg_fare + 2 * zone_std_fare
        ORDER BY z_score DESC;
""")

┌────────────┬───────────┬──────────────────────────────┬──────────┬───────────┬───────────────┬───────────────┬─────────┐
│    date    │   fare    │             zone             │ location │  borough  │ zone_avg_fare │ zone_std_fare │ z_score │
│    date    │  double   │           varchar            │  int64   │  varchar  │    double     │    double     │ double  │
├────────────┼───────────┼──────────────────────────────┼──────────┼───────────┼───────────────┼───────────────┼─────────┤
│ 2025-01-20 │ 863372.12 │ LaGuardia Airport            │      138 │ Queens    │         52.29 │        2953.6 │  292.29 │
│ 2025-01-16 │    2450.9 │ Midtown Center               │      161 │ Manhattan │         15.02 │         13.23 │  184.08 │
│ 2025-01-22 │     434.9 │ Upper East Side North        │      236 │ Manhattan │         12.51 │          8.23 │   51.35 │
│ 2025-01-09 │     600.0 │ Murray Hill                  │      170 │ Manhattan │         14.49 │         11.55 │   50.71 │
│ 2025-01-04 │  

*Calculate cumulative monthly revenue*

In [239]:
con.sql("""
    WITH monthly_revenue AS (
            SELECT
                SUM(fare_amount) as monthly_rev,
                DATE_TRUNC('month', pickup_datetime) AS month
            FROM taxi_clean
            WHERE fare_amount >= 0
            GROUP BY month
        )
        SELECT
            month,
            ROUND(monthly_rev, 2) AS monthly_revenue,
            ROUND(SUM(monthly_rev) OVER(ORDER BY month), 2) AS monthly_cumulative
        FROM monthly_revenue
        ORDER BY month;
""")

┌─────────────────────┬─────────────────┬────────────────────┐
│        month        │ monthly_revenue │ monthly_cumulative │
│      timestamp      │     double      │       double       │
├─────────────────────┼─────────────────┼────────────────────┤
│ 2024-12-01 00:00:00 │           401.6 │              401.6 │
│ 2025-01-01 00:00:00 │     51406146.12 │        51406547.72 │
│ 2025-02-01 00:00:00 │             6.5 │        51406554.22 │
└─────────────────────┴─────────────────┴────────────────────┘

*Rank routes by tip percentage*

In [ ]:
con.sql("""
    WITH pickup_zone AS (
            SELECT
                t.pickup_datetime AS pu_date,
                t.dropoff_datetime AS do_date,
                t.tip_amount AS tip,
                z.Zone AS zone
            FROM taxi_clean AS t
            INNER JOIN zone_lookup AS z
            ON t.pu_location_id = z.LocationID
            GROUP BY 
        )
        SELECT
            pu_date,
            do_date,
            tip,
            zone,
            RANK() OVER(
                ORDER BY tip DESC
            ) AS tip_rank
        FROM pickup_zone
        ORDER BY tip;
""")

┌─────────────────────┬─────────────────────┬────────┬───────────────────────────────┬──────────┐
│       pu_date       │       do_date       │  tip   │             zone              │ tip_rank │
│      timestamp      │      timestamp      │ double │            varchar            │  int64   │
├─────────────────────┼─────────────────────┼────────┼───────────────────────────────┼──────────┤
│ 2025-01-01 13:22:44 │ 2025-01-01 13:37:35 │    0.0 │ Lenox Hill West               │  2261857 │
│ 2025-01-22 12:17:37 │ 2025-01-22 12:28:49 │    0.0 │ Lincoln Square East           │  2261857 │
│ 2025-01-01 14:42:28 │ 2025-01-01 14:50:53 │    0.0 │ Canarsie                      │  2261857 │
│ 2025-01-22 12:12:59 │ 2025-01-22 12:59:05 │    0.0 │ East Harlem South             │  2261857 │
│ 2025-01-01 14:56:46 │ 2025-01-01 15:33:30 │    0.0 │ JFK Airport                   │  2261857 │
│ 2025-01-01 14:46:57 │ 2025-01-01 14:54:10 │    0.0 │ Midtown Center                │  2261857 │
│ 2025-01-01 16:04:4